# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides an interactive template for loading and exploring the [FAIR\^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. This loads both the dataset schema and prepares for record extraction.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant dataset schema URL (JSON-LD)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata

display_name = getattr(meta, 'name', None) or getattr(meta, 'title', None) or 'Dataset'
description = getattr(meta, 'description', '')
print(f"{display_name}: {description}")

## 2. Data Overview

Review available record sets and their metadata. Each `record_set` and `field` is identified by its Croissant `@id`, which is unique.

In this section, we'll enumerate the available record sets and describe their constituent fields to understand the structure of the dataset.

In [ ]:
# List available RecordSets and their fields by @id
record_sets = dataset.metadata.record_sets
if not record_sets:
    raise RuntimeError('No record sets found in the dataset!')

print(f"Total record sets: {len(record_sets)}")
print()
# Print overview
for rs in record_sets:
    print(f"- RecordSet: {rs.id}")
    if hasattr(rs, 'name'):
        print(f"  Name: {rs.name}")
    if hasattr(rs, 'description'):
        print(f"  Description: {rs.description}")
    if rs.fields:
        print("  Fields:")
        for f in rs.fields:
            print(f"    - {f.id} (name: {getattr(f, 'name', None)})")
    else:
        print("  No fields found.")
    print()
print("You can refer to the above @id values for record_set and field access.")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Below, select one or more record sets identified by their `@id` values above, and extract records to pandas DataFrames.

In [ ]:
# List of record_set @id strings to load (adjust as needed based on output above)
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

print(f'Will load {len(record_set_ids)} record sets into DataFrames:')
for rid in record_set_ids:
    print(f'- {rid}')

for record_set_id in record_set_ids:
    print(f"Loading records from: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Loaded {len(df)} records. Columns:")
    print(df.columns.tolist())
    print(df.head(2))


## 4. Exploratory Data Analysis (EDA)

Let's perform some exploratory data operations for a selected record set. We'll:

- Filter records based on a numeric field,
- Normalize a numeric field,
- Optionally, group by a categorical field.

**Important:** Use field `@id` for column selection. Choose meaningful numeric and group fields based on the overview above.

In [ ]:
# Choose a record set for further analysis
main_record_set_id = record_set_ids[0]
df = dataframes[main_record_set_id]

# List all columns to identify available fields
print("Available columns for analysis:")
print(df.columns.tolist())

# Select likely numeric and group fields based on their IDs (adjust as needed)
possible_numeric_fields = [c for c in df.columns if df[c].dtype in [int, float] or pd.api.types.is_numeric_dtype(df[c])]
if not possible_numeric_fields:
    possible_numeric_fields = [c for c in df.columns if 'count' in c or 'percent' in c or 'weight' in c or 'abundance' in c or 'MW' in c]

if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
else:
    raise RuntimeError('No numeric-like field found for demonstration.')

possible_group_fields = [c for c in df.columns if df[c].dtype == object and c != numeric_field_id]
group_field_id = possible_group_fields[0] if possible_group_fields else None

print(f"\nSelected numeric_field: {numeric_field_id}")
if group_field_id: print(f"Selected group_field: {group_field_id}")

# Filtering and normalization
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    threshold = df[numeric_field_id].mean() if pd.notna(df[numeric_field_id].mean()) else 0
else:
    try:
        # Try convert to float
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean() if pd.notna(df[numeric_field_id].mean()) else 0
    except:
        threshold = 0

filtered_df = df[df[numeric_field_id] > threshold]
print(f"\nFiltered DataFrame with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalization
filtered_df = filtered_df.copy()
filtered_df.loc[:, f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
)

print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped average {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())


## 5. Visualization

Let's plot the distribution of the numeric field and the grouped means, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of main numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
plt.title(f"Distribution of '{numeric_field_id}'")
plt.xlabel(numeric_field_id)
plt.show()

# If grouped_df exists, plot barplot of group_field vs mean value
if 'grouped_df' in locals() and group_field_id is not None and not grouped_df.empty:
    plt.figure(figsize=(9,4))
    sns.barplot(
        data=grouped_df,
        x=group_field_id,
        y=numeric_field_id
    )
    plt.xticks(rotation=45)
    plt.title(f"Mean of '{numeric_field_id}' grouped by '{group_field_id}'")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- The [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset is loaded and record sets can be explored by referencing `@id` values provided by the Croissant schema.
- We demonstrated programmatic loading, filtering, normalization, grouping, and visualization using field and record set `@id`.
- For deeper domain analysis, refer to biological specifics documented in the original Croissant metadata.